In [45]:
#librerias
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.stats import norm
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib
import os

In [46]:

# =========================
# 0) Load CSV from Drive
# =========================
#drive.mount("/content/drive")
CSV_PATH = "../results/fechas_117/TERMICA/inputRed2.csv"   # <-- adjust
df = pd.read_csv(
    CSV_PATH,
    sep=";",
    decimal="."     # si tienes 1.234.567,89
)
df.columns = df.columns.str.strip() 
df=df.sort_values(["Fecha", "id_planta"]).reset_index(drop=True)
required = ['id_planta', "available_q", "phi", "precio_d", "daily_reman_demand"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

# Optional: if present, enables per-player phi sampling
AGENT_COL = "id_planta" if "id_planta" in df.columns else None



In [47]:
df

,CodigoPlanta,available_q,phi,precio_d,Fecha,cluster,id_planta,daily_reman_demand
0,TYP3,"[49000.0, 49000.0, 49000.0, 49000.0, 49000.0,...",83.924097,101.37,2025-04-01,10.0,1,"[1029700.0, 935380.0, 843320.0, 771690.0, 695..."
1,TYP2,"[26000.0, 26000.0, 26000.0, 26000.0, 26000.0,...",262.221847,282.95,2025-04-01,,2,"[1029700.0, 935380.0, 843320.0, 771690.0, 695..."
2,TSJ2,"[170000.0, 170000.0, 170000.0, 170000.0, 1700...",275.288434,326.37,2025-04-01,10.0,3,"[1029700.0, 935380.0, 843320.0, 771690.0, 695..."
3,TSJ1,"[165000.0, 165000.0, 165000.0, 165000.0, 1650...",260.379933,310.37,2025-04-01,10.0,4,"[1029700.0, 935380.0, 843320.0, 771690.0, 695..."
4,MRL1,"[164000.0, 164000.0, 164000.0, 164000.0, 1640...",968.472039,1028.13,2025-04-01,10.0,5,"[1029700.0, 935380.0, 843320.0, 771690.0, 695..."
...,...,...,...,...,...,...,...,...
4207,TRN1,"[80000.0, 80000.0, 80000.0, 80000.0, 80000.0,...",779.362162,908.94,2025-09-09,12.0,32,"[1092150.0, 1009760.0, 945220.0, 868330.0, 80..."
4208,TBQ4,"[60000.0, 60000.0, 60000.0, 60000.0, 60000.0,...",1058.580711,1117.35,2025-09-09,12.0,33,"[1092150.0, 1009760.0, 945220.0, 868330.0, 80..."
4209,TBQ3,"[60000.0, 60000.0, 60000.0, 60000.0, 60000.0,...",945.489357,1012.74,2025-09-09,12.0,34,"[1092150.0, 1009760.0, 945220.0, 868330.0, 80..."
4210,TVL1,"[241000.0, 241000.0, 241000.0, 241000.0, 2410...",969.785873,1183.48,2025-09-09,12.0,35,"[1092150.0, 1009760.0, 945220.0, 868330.0, 80..."


## p fuel

In [48]:
df_desglose = pd.read_excel("../datos/raw/desglose.xlsx",header=3) #cambiar la direccion

In [49]:

df_desglose['Código SIC'] = df_desglose['Código SIC'].astype(str).str.strip()
df['CodigoPlanta'] = df['CodigoPlanta'].astype(str).str.strip()
df = df.merge(
    df_desglose[["Código SIC", 'Combustible por Defecto']],
    left_on="CodigoPlanta",
    right_on="Código SIC",
    how="left"
)
df = df.drop(columns=["Código SIC"])

In [52]:
df['Combustible por Defecto'].value_counts()

Combustible por Defecto
GAS                    2340
CARBON                 1521
ACPM                    234
MEZCLA GAS - CARBÓN     117
Name: count, dtype: int64